# 🗑️ WasteVision AI — Backend Notebook

This notebook contains the full WasteVision AI backend implemented in a single runnable `.ipynb` file.

**Sections:**
1. Install Dependencies
2. Project Structure Setup
3. Database (`db.py`)
4. Classifier Service (`classifier.py`)
5. YOLO Detector Service (`detector.py`)
6. OCR Engine Service (`ocr_engine.py`)
7. PDF Report Generator (`report_generator.py`)
8. FastAPI Routes (detect, ocr, history, report, webcam)
9. Main App & Launch
10. Training — Classifier
11. Training — YOLOv8

## 1. Install Dependencies

In [1]:
!pip install \
    fastapi==0.111.0 \
    uvicorn[standard]==0.29.0 \
    python-multipart==0.0.9 \
    torch==2.3.0 \
    torchvision==0.18.0 \
    ultralytics==8.2.0 \
    opencv-python-headless==4.9.0.80 \
    tensorflow==2.16.1 \
    easyocr==1.7.1 \
    Pillow==10.3.0 \
    aiofiles==23.2.1 \
    reportlab==4.1.0 \
    aiosqlite==0.20.0 \
    numpy==1.26.4 \
    nest_asyncio  

ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


In [ ]:
# ── Kaggle API Setup ─────────────────────────

import os
import json

# Your Kaggle credentials
os.environ["KAGGLE_USERNAME"] = "YOUR_USERNAME"
os.environ["KAGGLE_KEY"] = "YOUR_API_KEY"

print("✅ Kaggle API configured")

In [ ]:
# ── Download Waste Dataset ───────────────────

!kaggle datasets download -d techsash/waste-classification-data

In [ ]:
# ── Extract Dataset ──────────────────────────

import zipfile
import os

zip_path = "waste-classification-data.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("dataset")

print("✅ Dataset extracted")

# Check extracted folders
print("\n📂 Extracted contents:")

for item in os.listdir("dataset"):
    print(" -", item)

## 2. Project Structure Setup

Creates the required folders on disk.

In [2]:
import os

dirs = [
    "models",
    "uploads",
    "database",
    "datasets/waste",
    "datasets/waste_yolo",
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

print("✅ Folder structure ready:")
for d in dirs:
    print(f"   {d}/")

✅ Folder structure ready:
   models/
   uploads/
   database/
   datasets/waste/
   datasets/waste_yolo/


## 3. Database — `database/db.py`

Async SQLite via `aiosqlite`. Stores every detection result.

In [3]:
# ── database/db.py ────────────────────────────────────────────────────────────

import aiosqlite
import json
from datetime import datetime

DB_PATH = "database/wastevision.db"

CREATE_TABLE = """
CREATE TABLE IF NOT EXISTS detections (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    filename TEXT,
    waste_class TEXT,
    confidence REAL,
    recyclable INTEGER,
    hazard_level TEXT,
    bboxes TEXT,
    ocr_text TEXT,
    timestamp TEXT
)
"""

async def init_db():
    async with aiosqlite.connect(DB_PATH) as db:
        await db.execute(CREATE_TABLE)
        await db.commit()
    print("✅ Database initialized:", DB_PATH)

async def save_detection(data: dict):
    async with aiosqlite.connect(DB_PATH) as db:
        await db.execute(
            """INSERT INTO detections
               (filename, waste_class, confidence, recyclable, hazard_level, bboxes, ocr_text, timestamp)
               VALUES (?,?,?,?,?,?,?,?)""",
            (
                data.get("filename", ""),
                data.get("class", ""),
                data.get("confidence", 0.0),
                1 if data.get("recyclable") else 0,
                data.get("hazard_level", "low"),
                json.dumps(data.get("bboxes", [])),
                data.get("ocr_text", ""),
                datetime.utcnow().isoformat(),
            ),
        )
        await db.commit()

async def get_history(limit: int = 50):
    async with aiosqlite.connect(DB_PATH) as db:
        db.row_factory = aiosqlite.Row
        async with db.execute(
            "SELECT * FROM detections ORDER BY id DESC LIMIT ?", (limit,)
        ) as cursor:
            rows = await cursor.fetchall()
            return [dict(r) for r in rows]

# ── Quick test ────────────────────────────────────────────────────────────────
import asyncio
import nest_asyncio
nest_asyncio.apply()

asyncio.run(init_db())

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x0000024953C2ADC0> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x0000024953C2ADC0> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: c

✅ Database initialized: database/wastevision.db


## 4. Classifier Service — `services/classifier.py`

MobileNetV3-Small transfer-learning model with 10 waste classes.

In [ ]:
# ── services/classifier.py ────────────────────────────────────────────────────

import numpy as np
import os
from PIL import Image

CLASSES = [
    "plastic", "metal", "glass", "organic", "paper",
    "cardboard", "battery", "e-waste", "medical_waste", "hazardous",
]

RECYCLABLE = {"plastic", "metal", "glass", "paper", "cardboard"}

HAZARD_MAP = {
    "battery":       "high",
    "e-waste":       "high",
    "medical_waste": "high",
    "hazardous":     "critical",
    "organic":       "low",
    "paper":         "low",
    "cardboard":     "low",
    "plastic":       "low",
    "metal":         "low",
    "glass":         "medium",
}

MODEL_PATH = "models/classifier.h5"
_model = None


def _build_model():
    import tensorflow as tf
    base = tf.keras.applications.MobileNetV3Small(
        input_shape=(224, 224, 3),
        include_top=False,
        weights="imagenet",
    )
    base.trainable = False
    x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(len(CLASSES), activation="softmax")(x)
    model = tf.keras.Model(base.input, out)
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model


def load_classifier():
    global _model
    import tensorflow as tf
    if _model is not None:
        return _model
    if os.path.exists(MODEL_PATH):
        _model = tf.keras.models.load_model(MODEL_PATH)
        print("✅ Loaded classifier from", MODEL_PATH)
    else:
        print("⚠️  No trained model found — building untrained model (run training first).")
        _model = _build_model()
    return _model


def preprocess(image: Image.Image) -> np.ndarray:
    import tensorflow as tf
    img = image.convert("RGB").resize((224, 224))
    arr = np.array(img, dtype=np.float32)
    arr = tf.keras.applications.mobilenet_v3.preprocess_input(arr)
    return np.expand_dims(arr, 0)


def classify(image: Image.Image) -> dict:
    model = load_classifier()
    arr = preprocess(image)
    preds = model.predict(arr, verbose=0)[0]
    idx = int(np.argmax(preds))
    confidence = float(preds[idx])
    waste_class = CLASSES[idx]
    return {
        "class":       waste_class,
        "confidence":  round(confidence, 4),
        "recyclable":  waste_class in RECYCLABLE,
        "hazard_level": HAZARD_MAP.get(waste_class, "low"),
        "all_scores":  {c: round(float(p), 4) for c, p in zip(CLASSES, preds)},
    }

print("✅ Classifier module ready. Call load_classifier() to load weights.")

## 5. YOLO Detector Service — `services/detector.py`

YOLOv8-nano for object/bounding-box detection.

In [ ]:
# ── services/detector.py ──────────────────────────────────────────────────────

import numpy as np
from PIL import Image
import os

_yolo = None
YOLO_MODEL = "models/yolov8n.pt"


def load_yolo():
    global _yolo
    if _yolo is not None:
        return _yolo
    from ultralytics import YOLO
    path = YOLO_MODEL if os.path.exists(YOLO_MODEL) else "yolov8n.pt"
    print(f"Loading YOLO from: {path}")
    _yolo = YOLO(path)
    return _yolo


def detect_objects(image: Image.Image) -> list:
    model = load_yolo()
    results = model(image, verbose=False)[0]
    boxes = []
    for box in results.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        conf = float(box.conf[0])
        cls_id = int(box.cls[0])
        label = results.names[cls_id]
        boxes.append({
            "label":      label,
            "confidence": round(conf, 4),
            "bbox":       [round(x1), round(y1), round(x2), round(y2)],
        })
    return boxes

print("✅ Detector module ready. Call load_yolo() to load weights.")

## 6. OCR Engine — `services/ocr_engine.py`

Text extraction from images using EasyOCR.

In [ ]:
# ── services/ocr_engine.py ────────────────────────────────────────────────────

import numpy as np
from PIL import Image

_reader = None


def load_reader():
    global _reader
    if _reader is None:
        import easyocr
        print("Loading EasyOCR (first run downloads ~100MB model)...")
        _reader = easyocr.Reader(["en"], gpu=False)
        print("✅ EasyOCR ready")
    return _reader


def extract_text(image: Image.Image) -> dict:
    reader = load_reader()
    arr = np.array(image.convert("RGB"))
    results = reader.readtext(arr)
    texts = [{"text": t, "confidence": round(c, 4)} for (_, t, c) in results]
    combined = " ".join(r["text"] for r in texts)
    return {"text": combined, "details": texts}

print("✅ OCR engine module ready.")

## 7. PDF Report Generator — `services/report_generator.py`

Generates a formatted PDF summary of all detections using ReportLab.

In [ ]:
# ── services/report_generator.py ─────────────────────────────────────────────

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from datetime import datetime
import io


def generate_pdf(detections: list) -> bytes:
    buf = io.BytesIO()
    doc = SimpleDocTemplate(buf, pagesize=A4)
    styles = getSampleStyleSheet()
    story = []

    story.append(Paragraph("WasteVision AI — Detection Report", styles["Title"]))
    story.append(Paragraph(
        f"Generated: {datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')}",
        styles["Normal"]
    ))
    story.append(Spacer(1, 20))

    headers = ["#", "Class", "Confidence", "Recyclable", "Hazard", "Timestamp"]
    rows = [headers]
    for i, d in enumerate(detections, 1):
        rows.append([
            str(i),
            d.get("waste_class", ""),
            f"{d.get('confidence', 0):.2%}",
            "Yes" if d.get("recyclable") else "No",
            d.get("hazard_level", ""),
            d.get("timestamp", "")[:19],
        ])

    t = Table(rows, repeatRows=1)
    t.setStyle(TableStyle([
        ("BACKGROUND",   (0, 0), (-1,  0), colors.HexColor("#1a3a2a")),
        ("TEXTCOLOR",    (0, 0), (-1,  0), colors.white),
        ("FONTNAME",     (0, 0), (-1,  0), "Helvetica-Bold"),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#f0f8f4")]),
        ("GRID",         (0, 0), (-1, -1), 0.5, colors.grey),
        ("FONTSIZE",     (0, 0), (-1, -1), 9),
    ]))
    story.append(t)

    total = len(detections)
    recyclable = sum(1 for d in detections if d.get("recyclable"))
    story.append(Spacer(1, 20))
    story.append(Paragraph(f"Total Detections: {total}", styles["Normal"]))
    story.append(Paragraph(
        f"Recyclable: {recyclable} ({recyclable / max(total, 1):.0%})",
        styles["Normal"]
    ))

    doc.build(story)
    return buf.getvalue()

print("✅ PDF report generator ready.")

## 8. FastAPI Routes

All five routes defined in a single cell for notebook convenience.

In [ ]:
# ── routes/detect.py ──────────────────────────────────────────────────────────

from fastapi import APIRouter, UploadFile, File, HTTPException
from PIL import Image
import io, os, uuid
import aiofiles

detect_router = APIRouter()
UPLOAD_DIR = "uploads"
os.makedirs(UPLOAD_DIR, exist_ok=True)


@detect_router.post("/detect")
async def detect(file: UploadFile = File(...)):
    if not file.content_type.startswith("image/"):
        raise HTTPException(400, "File must be an image")

    data = await file.read()
    image = Image.open(io.BytesIO(data))

    filename = f"{uuid.uuid4().hex}_{file.filename}"
    path = os.path.join(UPLOAD_DIR, filename)
    async with aiofiles.open(path, "wb") as f:
        await f.write(data)

    clf = classify(image)
    bboxes = detect_objects(image)

    result = {
        "filename":    filename,
        "class":       clf["class"],
        "confidence":  clf["confidence"],
        "recyclable":  clf["recyclable"],
        "hazard_level": clf["hazard_level"],
        "all_scores":  clf["all_scores"],
        "bboxes":      bboxes,
    }
    await save_detection(result)
    return result


# ── routes/ocr.py ─────────────────────────────────────────────────────────────

ocr_router = APIRouter()


@ocr_router.post("/ocr")
async def ocr(file: UploadFile = File(...)):
    if not file.content_type.startswith("image/"):
        raise HTTPException(400, "File must be an image")
    data = await file.read()
    image = Image.open(io.BytesIO(data))
    return extract_text(image)


# ── routes/history.py ─────────────────────────────────────────────────────────

from fastapi import Query

history_router = APIRouter()


@history_router.get("/history")
async def history(limit: int = Query(50, ge=1, le=500)):
    rows = await get_history(limit)
    return {"count": len(rows), "detections": rows}


# ── routes/report.py ──────────────────────────────────────────────────────────

from fastapi.responses import Response

report_router = APIRouter()


@report_router.get("/report/pdf")
async def pdf_report(limit: int = 100):
    detections = await get_history(limit)
    pdf_bytes = generate_pdf(detections)
    return Response(
        content=pdf_bytes,
        media_type="application/pdf",
        headers={"Content-Disposition": "attachment; filename=wastevision-report.pdf"},
    )


# ── routes/webcam.py ──────────────────────────────────────────────────────────

from fastapi import WebSocket, WebSocketDisconnect
import base64, json

webcam_router = APIRouter()


@webcam_router.websocket("/ws/webcam")
async def webcam_ws(websocket: WebSocket):
    await websocket.accept()
    try:
        while True:
            msg = await websocket.receive_text()
            payload = json.loads(msg)

            img_b64 = payload.get("image", "")
            if "," in img_b64:
                img_b64 = img_b64.split(",", 1)[1]

            img_bytes = base64.b64decode(img_b64)
            image = Image.open(io.BytesIO(img_bytes))

            clf    = classify(image)
            bboxes = detect_objects(image)

            result = {
                "class":       clf["class"],
                "confidence":  clf["confidence"],
                "recyclable":  clf["recyclable"],
                "hazard_level": clf["hazard_level"],
                "bboxes":      bboxes,
            }
            await save_detection({**result, "filename": "webcam"})
            await websocket.send_text(json.dumps(result))

    except WebSocketDisconnect:
        pass
    except Exception as e:
        await websocket.send_text(json.dumps({"error": str(e)}))
        await websocket.close()

print("✅ All routes defined: detect_router, ocr_router, history_router, report_router, webcam_router")

## 9. Main FastAPI App & Launch

Combines all routers and starts the server with `uvicorn` inside the notebook.

> **API Docs** will be at → http://localhost:8000/docs

In [ ]:
# ── main.py ───────────────────────────────────────────────────────────────────

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from contextlib import asynccontextmanager


@asynccontextmanager
async def lifespan(app: FastAPI):
    await init_db()
    yield


app = FastAPI(title="WasteVision AI", lifespan=lifespan)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

app.include_router(detect_router)
app.include_router(ocr_router)
app.include_router(history_router)
app.include_router(report_router)
app.include_router(webcam_router)


@app.get("/")
async def root():
    return {"status": "WasteVision AI running"}


print("✅ FastAPI app created with all routes.")
print()
print("Routes registered:")
for route in app.routes:
    if hasattr(route, 'methods'):
        print(f"  {list(route.methods)} {route.path}")
    elif hasattr(route, 'path'):
        print(f"  [WS] {route.path}")

In [ ]:
# ── Launch server ─────────────────────────────────────────────────────────────
# This cell starts the FastAPI server.
# ⚠️  It will block this cell — open API docs in a new tab: http://localhost:8000/docs
# To stop: Kernel → Interrupt

import uvicorn

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()

## 10. Training — Classifier (MobileNetV3)

Run this section **only** when you have a dataset. Place images in:
```
datasets/waste/{class_name}/*.jpg
```
Supported class names: `plastic metal glass organic paper cardboard battery e-waste medical_waste hazardous`

In [ ]:
# ── training/train_classifier.py ──────────────────────────────────────────────
# Configure these before running:

DATA_DIR         = "datasets/waste"  # ImageFolder root
EPOCHS           = 15
FINE_TUNE_EPOCHS = 10
BATCH            = 32
IMG_SIZE         = (224, 224)
MODEL_OUT        = "models/classifier.h5"

# ─────────────────────────────────────────────────────────────────────────────

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

os.makedirs("models", exist_ok=True)

train_gen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v3.preprocess_input,
    validation_split=0.2,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2,
)

train = train_gen.flow_from_directory(
    DATA_DIR, target_size=IMG_SIZE, batch_size=BATCH, subset="training"
)
val = train_gen.flow_from_directory(
    DATA_DIR, target_size=IMG_SIZE, batch_size=BATCH, subset="validation"
)

num_classes = len(train.class_indices)
print(f"Classes found: {train.class_indices}")

# Build model
base = tf.keras.applications.MobileNetV3Small(
    input_shape=(*IMG_SIZE, 3), include_top=False, weights="imagenet"
)
base.trainable = False
x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
x = tf.keras.layers.Dense(256, activation="relu")(x)
x = tf.keras.layers.Dropout(0.3)(x)
out = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
model = tf.keras.Model(base.input, out)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

# Phase 1 — head only
print(f"\n🔵 Phase 1: Training head — {EPOCHS} epochs")
model.fit(train, validation_data=val, epochs=EPOCHS)

# Phase 2 — fine-tune last 30 base layers
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

print(f"\n🟢 Phase 2: Fine-tuning — {FINE_TUNE_EPOCHS} epochs")
model.fit(
    train,
    validation_data=val,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(MODEL_OUT, save_best_only=True),
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    ],
)

model.save(MODEL_OUT)
print(f"\n✅ Classifier saved → {MODEL_OUT}")
print("Class mapping:", train.class_indices)

## 11. Training — YOLOv8

Dataset must be in YOLO format:
```
datasets/waste_yolo/
    images/train/   images/val/
    labels/train/   labels/val/
    data.yaml
```
Download datasets from Kaggle (see README links) and extract into `datasets/`.

In [ ]:
# ── training/train_yolo.py ────────────────────────────────────────────────────
# Configure these before running:

DATA_YAML  = "datasets/waste_yolo/data.yaml"
YOLO_EPOCHS = 50
IMGSZ      = 640

# ─────────────────────────────────────────────────────────────────────────────

from ultralytics import YOLO

model = YOLO("yolov8n.pt")   # downloads automatically on first run

results = model.train(
    data=DATA_YAML,
    epochs=YOLO_EPOCHS,
    imgsz=IMGSZ,
    project="models",
    name="waste_yolo",
    exist_ok=True,
)

print("✅ YOLOv8 training complete!")
print("Best weights saved at: models/waste_yolo/weights/best.pt")
print("Copy that file to models/yolov8n.pt to use it in the API.")

## 12. Quick Smoke Test (optional)

Test classify + OCR on a local image without starting the server.

In [ ]:
from PIL import Image as PILImage
import requests
from io import BytesIO

# Download a sample image for testing
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/320px-Cat03.jpg"
response = requests.get(url)
test_image = PILImage.open(BytesIO(response.content))

print("=== Classifier ===")
clf_result = classify(test_image)
print(clf_result)

print("\n=== Object Detection ===")
bbox_result = detect_objects(test_image)
print(bbox_result)

print("\n=== OCR ===")
ocr_result = extract_text(test_image)
print(ocr_result)